# Processing Raw Conversion

This notebook converts the compressed eICU files from:
```
data/raw/eicu/*.csv.gz
```
into standard CSV files stored in:
```
data/processed/*.csv
```
It also checks the converted files, reads their column names, and summarizes their sizes.

### Import Required Libraries

In [1]:
from pathlib import Path
import gzip
import shutil

import pandas as pd
from tqdm.auto import tqdm
from IPython.display import display

#### Explanation :
- `Path` manages project folders and file paths.
- `gzip` opens compressed .csv.gz files.
- `shutil` copies the decompressed data into .csv files.
- `pandas` creates summary tables.
- `tqdm` shows progress while files are being processed.
- `display` presents DataFrames clearly inside the notebook

### Locate the Project Root Folder

In [2]:
def find_project_root(start_path):
    """
    Search upward from the current folder until the project root is found.

    The project root must contain:
    - data/
    - notebooks/
    """
    
    current_path = start_path.resolve()

    for folder in [current_path, *current_path.parents]:
        data_folder = folder / "data"
        notebooks_folder = folder / "notebooks"

        if data_folder.is_dir() and notebooks_folder.is_dir():
            return folder

    raise FileNotFoundError(
        "Project root could not be found. "
        "Make sure the project contains both data/ and notebooks/ folders."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

print(f"Project root: {PROJECT_ROOT}")

Project root: C:\Users\samsa\Documents\Capstone Project\Project


#### Explanation :
The notebook is stored inside:
```
notebooks/module_1/
```
The function searches upward from the current working folder until it finds a folder containing both `data` and `notebooks`.

This is safer than using a fixed number of `.parent` operations because the notebook can work whether Jupyter starts from the project root, `notebooks`, or `notebooks/module_1`.

### Define the Raw and Processed Data Folders

In [3]:
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "eicu"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

print("=" * 100)
print("PROJECT FOLDER PATHS")
print("=" * 100)
print(f"Project root     : {PROJECT_ROOT}")
print(f"Raw eICU folder  : {RAW_DIR}")
print(f"Processed folder : {PROCESSED_DIR}")
print("=" * 100)

PROJECT FOLDER PATHS
Project root     : C:\Users\samsa\Documents\Capstone Project\Project
Raw eICU folder  : C:\Users\samsa\Documents\Capstone Project\Project\data\raw\eicu
Processed folder : C:\Users\samsa\Documents\Capstone Project\Project\data\processed


#### Explanation :
The notebook uses the following folders:

- Raw compressed files: `data/raw/eicu/`
- Converted CSV files: `data/processed/`

Using `Path` keeps the code readable and works across Windows, macOS, and Linux.

### Check the Input Folder and Create the Output Folder

In [4]:
if not RAW_DIR.exists():
    raise FileNotFoundError(
        f"Raw eICU folder was not found:\n{RAW_DIR}"
    )

if not RAW_DIR.is_dir():
    raise NotADirectoryError(
        f"The raw eICU path is not a folder:\n{RAW_DIR}"
    )

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Raw eICU folder found.")
print("Processed folder is ready.")

Raw eICU folder found.
Processed folder is ready.


#### Explanation :
The code stops with a clear error if `data/raw/eicu/` does not exist.

The `data/processed/` folder is created automatically when needed. Existing folders and files are not deleted.

### Find All Compressed eICU Files

In [5]:
gz_files = sorted(RAW_DIR.glob("*.csv.gz"))

print(f"Total .csv.gz files found: {len(gz_files)}")

if len(gz_files) == 0:
    raise FileNotFoundError(
        f"No .csv.gz files were found inside:\n{RAW_DIR}"
    )

Total .csv.gz files found: 31


#### Explanation :
This code searches the raw eICU folder for files ending in `.csv.gz`.

The files are sorted by name so that they are processed in a consistent order.

### Display the Raw File Names

In [6]:
print("=" * 100)
print("RAW eICU FILES")
print("=" * 100)

for file_number, gz_file in enumerate(gz_files, start=1):
    print(f"{file_number:02d}. {gz_file.name}")

RAW eICU FILES
01. admissionDrug.csv.gz
02. admissionDx.csv.gz
03. allergy.csv.gz
04. apacheApsVar.csv.gz
05. apachePatientResult.csv.gz
06. apachePredVar.csv.gz
07. carePlanCareProvider.csv.gz
08. carePlanEOL.csv.gz
09. carePlanGeneral.csv.gz
10. carePlanGoal.csv.gz
11. carePlanInfectiousDisease.csv.gz
12. customLab.csv.gz
13. diagnosis.csv.gz
14. hospital.csv.gz
15. infusionDrug.csv.gz
16. intakeOutput.csv.gz
17. lab.csv.gz
18. medication.csv.gz
19. microLab.csv.gz
20. note.csv.gz
21. nurseAssessment.csv.gz
22. nurseCare.csv.gz
23. nurseCharting.csv.gz
24. pastHistory.csv.gz
25. patient.csv.gz
26. physicalExam.csv.gz
27. respiratoryCare.csv.gz
28. respiratoryCharting.csv.gz
29. treatment.csv.gz
30. vitalAperiodic.csv.gz
31. vitalPeriodic.csv.gz


#### Explanation :
This cell displays every compressed eICU table before conversion.

It helps confirm that files such as `patient.csv.gz`, `lab.csv.gz`, and `vitalPeriodic.csv.gz` are available.

### Choose Whether Existing CSV Files Should Be Replaced

In [7]:
OVERWRITE_EXISTING = False

print(f"Overwrite existing CSV files: {OVERWRITE_EXISTING}")

Overwrite existing CSV files: False


#### Explanation :
When `OVERWRITE_EXISTING` is `False`, an existing processed CSV file is skipped.

Set it to `True` only when all processed files should be recreated from the compressed raw files.

### Convert the Compressed Files into CSV Files

In [8]:
conversion_records = []

for gz_file in tqdm(
    gz_files,
    desc="Converting eICU files",
    unit="file"
):
    # admissionDrug.csv.gz becomes admissionDrug.csv
    csv_file = PROCESSED_DIR / gz_file.with_suffix("").name

    # Temporary file prevents an incomplete CSV from replacing a valid file
    temporary_file = csv_file.with_suffix(".csv.tmp")

    status = ""
    error_message = ""

    try:
        if csv_file.exists() and not OVERWRITE_EXISTING:
            status = "Skipped"

        else:
            # Remove an old temporary file if one exists
            if temporary_file.exists():
                temporary_file.unlink()

            # Decompress the source file into the temporary CSV file
            with gzip.open(gz_file, "rb") as compressed_file:
                with open(temporary_file, "wb") as output_file:
                    shutil.copyfileobj(compressed_file, output_file)

            # Replace the final CSV only after conversion succeeds
            temporary_file.replace(csv_file)

            status = "Converted"

    except Exception as error:
        status = "Failed"
        error_message = str(error)

        # Remove an incomplete temporary file
        if temporary_file.exists():
            temporary_file.unlink()

    conversion_records.append({
        "raw_file": gz_file.name,
        "processed_file": csv_file.name,
        "status": status,
        "error": error_message
    })

Converting eICU files:   0%|          | 0/31 [00:00<?, ?file/s]

#### Explanation :
Each `.csv.gz` file is decompressed and saved as a regular `.csv` file.

For example:
```
data/raw/eicu/patient.csv.gz
```
becomes:
```
data/processed/patient.csv
```
The data is copied in smaller blocks instead of loading the entire table into memory. This is important because some eICU tables are very large.

A temporary file is used during conversion. The final CSV is created only after the decompression finishes successfully.

### Review the Conversion Results

In [9]:
conversion_summary_df = pd.DataFrame(conversion_records)

display(conversion_summary_df)

,raw_file,processed_file,status,error
0,admissionDrug.csv.gz,admissionDrug.csv,Converted,
1,admissionDx.csv.gz,admissionDx.csv,Converted,
2,allergy.csv.gz,allergy.csv,Converted,
3,apacheApsVar.csv.gz,apacheApsVar.csv,Converted,
4,apachePatientResult.csv.gz,apachePatientResult.csv,Converted,
5,apachePredVar.csv.gz,apachePredVar.csv,Converted,
6,carePlanCareProvider.csv.gz,carePlanCareProvider.csv,Converted,
7,carePlanEOL.csv.gz,carePlanEOL.csv,Converted,
8,carePlanGeneral.csv.gz,carePlanGeneral.csv,Converted,
9,carePlanGoal.csv.gz,carePlanGoal.csv,Converted,


#### Explanation :
The summary table shows the result for every raw file:

- `Converted`: the CSV file was created.
- `Skipped`: the CSV file already existed.
- `Failed`: an error occurred during conversion.

The `error` column contains the error message when a conversion fails.

### Count Coverted, Skipped and Failed Files

In [10]:
converted_count = (
    conversion_summary_df["status"] == "Converted"
).sum()

skipped_count = (
    conversion_summary_df["status"] == "Skipped"
).sum()

failed_count = (
    conversion_summary_df["status"] == "Failed"
).sum()

print("=" * 70)
print("CONVERSION SUMMARY")
print("=" * 70)
print(f"Raw files found : {len(gz_files)}")
print(f"Files converted : {converted_count}")
print(f"Files skipped   : {skipped_count}")
print(f"Files failed    : {failed_count}")
print("=" * 70)

CONVERSION SUMMARY
Raw files found : 31
Files converted : 31
Files skipped   : 0
Files failed    : 0


#### Explanation :
This cell gives a simple count of the conversion results.

Skipped files are not errors. They mean that the processed CSV files were already available and `OVERWRITE_EXISTING` was set to `False`.

### Display Failed Conversions

In [11]:
failed_files_df = conversion_summary_df[
    conversion_summary_df["status"] == "Failed"
].copy()

if failed_files_df.empty:
    print("No conversion failures were found.")
else:
    print("The following files could not be converted:")
    display(failed_files_df)

No conversion failures were found.


#### Explanation :
Only failed conversions are displayed in this section.

This makes it easier to identify a damaged compressed file, permission problem, or storage issue.

### Find All Processed CSV Files

In [12]:
csv_files = sorted(PROCESSED_DIR.glob("*.csv"))

print(f"Total CSV files in data/processed: {len(csv_files)}")

if len(csv_files) == 0:
    raise FileNotFoundError(
        f"No CSV files were found inside:\n{PROCESSED_DIR}"
    )

Total CSV files in data/processed: 31


#### Explanation :
After conversion, the code searches `data/processed/` for all CSV files.

These files are used in the remaining validation and summary steps.

### Verify That Every Raw File Has a Processed CSV File

In [13]:
expected_csv_names = {
    gz_file.with_suffix("").name
    for gz_file in gz_files
}

processed_csv_names = {
    csv_file.name
    for csv_file in csv_files
}

missing_csv_files = sorted(
    expected_csv_names - processed_csv_names
)

additional_csv_files = sorted(
    processed_csv_names - expected_csv_names
)

print("=" * 100)
print("FILE MATCHING CHECK")
print("=" * 100)
print(f"Expected CSV files   : {len(expected_csv_names)}")
print(f"Processed CSV files  : {len(processed_csv_names)}")
print(f"Missing CSV files    : {len(missing_csv_files)}")
print(f"Additional CSV files : {len(additional_csv_files)}")
print("=" * 100)

FILE MATCHING CHECK
Expected CSV files   : 31
Processed CSV files  : 31
Missing CSV files    : 0
Additional CSV files : 0


#### Explanation :
The expected CSV names are created from the raw file names.

For example:
```
lab.csv.gz → lab.csv
```
The code compares the expected names with the files found in `data/processed/`.

An additional file is not necessarily a problem. It may be a CSV that was placed in the processed folder separately.

### Display Missing Processed Files

In [14]:
if missing_csv_files:
    print("Raw files without matching processed CSV files:")

    for file_name in missing_csv_files:
        print(f"- {file_name}")
else:
    print("Every raw .csv.gz file has a matching processed CSV file.")

Every raw .csv.gz file has a matching processed CSV file.


#### Explanation :
This cell identifies compressed files that do not have a matching CSV file.

A missing file may indicate that its conversion failed or that the notebook stopped before conversion finished.

### Display Additional Processed Files

In [15]:
if additional_csv_files:
    print("Additional CSV files found in data/processed:")

    for file_name in additional_csv_files:
        print(f"- {file_name}")
else:
    print("No additional CSV files were found.")

No additional CSV files were found.


#### Explanation :
This section displays CSV files that are present in `data/processed/` but do not have a matching `.csv.gz` file in `data/raw/eicu/`.

The notebook does not delete or modify these additional files.

### Read the Column Names from Every Processed Table

In [16]:
file_column_summary = []
column_read_errors = []

for csv_file in tqdm(
    csv_files,
    desc="Reading table headers",
    unit="file"
):
    try:
        # Read only the header, not the full table
        header_df = pd.read_csv(
            csv_file,
            nrows=0,
            low_memory=False
        )

        column_names = header_df.columns.tolist()

        file_column_summary.append({
            "file_name": csv_file.name,
            "total_columns": len(column_names),
            "column_names": column_names
        })

    except Exception as error:
        column_read_errors.append({
            "file_name": csv_file.name,
            "error": str(error)
        })

Reading table headers:   0%|          | 0/31 [00:00<?, ?file/s]

#### Explanation :
Only the header row of each CSV file is read.

This provides the column names without loading millions of patient records into memory. It is safer and faster for large eICU tables.

Any table that cannot be read is stored in a separate error list.

### Create the Column Summary Table

In [17]:
column_summary_df = pd.DataFrame(file_column_summary)

if not column_summary_df.empty:
    column_summary_df = column_summary_df.sort_values(
        by="file_name"
    ).reset_index(drop=True)

display(column_summary_df)

,file_name,total_columns,column_names
0,admissionDrug.csv,14,"[admissiondrugid, patientunitstayid, drugoffse..."
1,admissionDx.csv,6,"[admissiondxid, patientunitstayid, admitdxente..."
2,allergy.csv,13,"[allergyid, patientunitstayid, allergyoffset, ..."
3,apacheApsVar.csv,26,"[apacheapsvarid, patientunitstayid, intubated,..."
4,apachePatientResult.csv,23,"[apachepatientresultsid, patientunitstayid, ph..."
5,apachePredVar.csv,51,"[apachepredvarid, patientunitstayid, sicuday, ..."
6,carePlanCareProvider.csv,8,"[cplcareprovderid, patientunitstayid, careprov..."
7,carePlanEOL.csv,5,"[cpleolid, patientunitstayid, cpleolsaveoffset..."
8,carePlanGeneral.csv,6,"[cplgeneralid, patientunitstayid, activeupondi..."
9,carePlanGoal.csv,7,"[cplgoalid, patientunitstayid, cplgoaloffset, ..."


#### Explanation :
The summary contains:

- The table filename
- The number of columns
- The complete list of column names

This gives a quick overview of the structure of every processed eICU table.

### Display Out the File Name and Column Count

In [18]:
if not column_summary_df.empty:
    display(
        column_summary_df[
            ["file_name", "total_columns"]
        ]
    )
else:
    print("No column information is available.")

,file_name,total_columns
0,admissionDrug.csv,14
1,admissionDx.csv,6
2,allergy.csv,13
3,apacheApsVar.csv,26
4,apachePatientResult.csv,23
5,apachePredVar.csv,51
6,carePlanCareProvider.csv,8
7,carePlanEOL.csv,5
8,carePlanGeneral.csv,6
9,carePlanGoal.csv,7


#### Explanation :
This is a shorter version of the column summary.

It is useful when only the number of variables in each eICU table is needed.

#### Display the Columns of Each Table Clearly

In [19]:
for table_information in file_column_summary:
    print("=" * 100)
    print(f"File: {table_information['file_name']}")
    print(f"Total columns: {table_information['total_columns']}")
    print("-" * 100)

    for column_number, column_name in enumerate(
        table_information["column_names"],
        start=1
    ):
        print(f"{column_number:02d}. {column_name}")

    print()

File: admissionDrug.csv
Total columns: 14
----------------------------------------------------------------------------------------------------
01. admissiondrugid
02. patientunitstayid
03. drugoffset
04. drugenteredoffset
05. drugnotetype
06. specialtytype
07. usertype
08. rxincluded
09. writtenineicu
10. drugname
11. drugdosage
12. drugunit
13. drugadmitfrequency
14. drughiclseqno

File: admissionDx.csv
Total columns: 6
----------------------------------------------------------------------------------------------------
01. admissiondxid
02. patientunitstayid
03. admitdxenteredoffset
04. admitdxpath
05. admitdxname
06. admitdxtext

File: allergy.csv
Total columns: 13
----------------------------------------------------------------------------------------------------
01. allergyid
02. patientunitstayid
03. allergyoffset
04. allergyenteredoffset
05. allergynotetype
06. specialtytype
07. usertype
08. rxincluded
09. writtenineicu
10. drugname
11. allergytype
12. allergyname
13. drughiclseq

#### Explanation :
This cell displays each table separately and numbers its columns.

The output is easier to read than showing every column list inside one large DataFrame.

### Display Tables That Could Not Be Read

In [20]:
column_error_df = pd.DataFrame(column_read_errors)

if column_error_df.empty:
    print("The headers of all processed CSV files were read successfully.")
else:
    print("The following CSV headers could not be read:")
    display(column_error_df)

The headers of all processed CSV files were read successfully.


#### Explanation :
This section checks whether every processed file has a readable CSV header.

A failure may indicate that a file is incomplete, damaged, empty, or not formatted as a valid CSV file.

### Calculate the Size of Every Processed File

In [21]:
file_size_summary = []

for csv_file in tqdm(
    csv_files,
    desc="Checking processed file sizes",
    unit="file"
):
    file_size_bytes = csv_file.stat().st_size
    file_size_mb = file_size_bytes / (1024 ** 2)
    file_size_gb = file_size_bytes / (1024 ** 3)

    file_size_summary.append({
        "file_name": csv_file.name,
        "file_path": str(csv_file),
        "size_bytes": file_size_bytes,
        "size_mb": round(file_size_mb, 2),
        "size_gb": round(file_size_gb, 4)
    })

Checking processed file sizes:   0%|          | 0/31 [00:00<?, ?file/s]

#### Explaination

The code measures each processed file in:

- Bytes
- Megabytes
- Gigabytes

This helps identify the largest eICU tables before later data cleaning and clustering work.

### Create the File Size Summary Table

In [23]:
file_size_summary_df = pd.DataFrame(file_size_summary)

file_size_summary_df = file_size_summary_df.sort_values(
    by="file_name"
).reset_index(drop=True)

display(file_size_summary_df)

,file_name,file_path,size_bytes,size_mb,size_gb
0,admissionDrug.csv,C:\Users\samsa\Documents\Capstone Project\Proj...,311224613,296.81,0.2899
1,admissionDx.csv,C:\Users\samsa\Documents\Capstone Project\Proj...,92417055,88.14,0.0861
2,allergy.csv,C:\Users\samsa\Documents\Capstone Project\Proj...,125620366,119.80,0.1170
3,apacheApsVar.csv,C:\Users\samsa\Documents\Capstone Project\Proj...,15843531,15.11,0.0148
4,apachePatientResult.csv,C:\Users\samsa\Documents\Capstone Project\Proj...,51729916,49.33,0.0482
5,apachePredVar.csv,C:\Users\samsa\Documents\Capstone Project\Proj...,22305326,21.27,0.0208
6,carePlanCareProvider.csv,C:\Users\samsa\Documents\Capstone Project\Proj...,27866190,26.58,0.0260
7,carePlanEOL.csv,C:\Users\samsa\Documents\Capstone Project\Proj...,38864,0.04,0.0000
8,carePlanGeneral.csv,C:\Users\samsa\Documents\Capstone Project\Proj...,186123017,177.50,0.1733
9,carePlanGoal.csv,C:\Users\samsa\Documents\Capstone Project\Proj...,37362797,35.63,0.0348


#### Explanation :

This table provides the location and storage size of every processed CSV file.

The files are initially arranged alphabetically.

### Display Files from Largest to Smallest

In [24]:
largest_files_df = file_size_summary_df.sort_values(
    by="size_bytes",
    ascending=False
).reset_index(drop=True)

display(
    largest_files_df[
        ["file_name", "size_mb", "size_gb"]
    ]
)

,file_name,size_mb,size_gb
0,nurseCharting.csv,10934.87,10.6786
1,vitalPeriodic.csv,7570.53,7.3931
2,lab.csv,2254.71,2.2019
3,nurseAssessment.csv,2231.54,2.1792
4,intakeOutput.csv,1853.86,1.8104
5,physicalExam.csv,1303.78,1.2732
6,respiratoryCharting.csv,1173.46,1.1460
7,nurseCare.csv,1129.30,1.1028
8,vitalAperiodic.csv,936.41,0.9145
9,medication.csv,603.64,0.5895


#### Explanation :
This section sorts the eICU tables from largest to smallest.

Large tables such as periodic vital signs can require more memory and processing time during later analysis.

### Calculate the Total Size of the Processed Data

In [25]:
total_size_bytes = file_size_summary_df["size_bytes"].sum()
total_size_mb = total_size_bytes / (1024 ** 2)
total_size_gb = total_size_bytes / (1024 ** 3)

print("=" * 70)
print("PROCESSED DATA SIZE")
print("=" * 70)
print(f"Total processed files : {len(file_size_summary_df)}")
print(f"Total size in bytes   : {total_size_bytes:,}")
print(f"Total size in MB      : {total_size_mb:,.2f} MB")
print(f"Total size in GB      : {total_size_gb:,.2f} GB")
print("=" * 70)

PROCESSED DATA SIZE
Total processed files : 31
Total size in bytes   : 33,876,002,241
Total size in MB      : 32,306.67 MB
Total size in GB      : 31.55 GB


#### Explanation :
This cell calculates the combined storage size of all CSV files in data/processed/.

It provides a clear estimate of the total processed dataset size.

### Perform the Final Validation

In [26]:
conversion_is_complete = (
    failed_count == 0
    and len(missing_csv_files) == 0
    and len(column_read_errors) == 0
)

print("=" * 70)
print("FINAL VALIDATION")
print("=" * 70)

if conversion_is_complete:
    print("Raw-to-processed conversion completed successfully.")
    print("Every raw file has a matching processed CSV file.")
    print("All processed CSV headers were read successfully.")
else:
    print("The conversion requires attention.")

    if failed_count > 0:
        print(f"- Failed conversions: {failed_count}")

    if missing_csv_files:
        print(f"- Missing processed CSV files: {len(missing_csv_files)}")

    if column_read_errors:
        print(f"- Unreadable CSV headers: {len(column_read_errors)}")

print("=" * 70)

FINAL VALIDATION
Raw-to-processed conversion completed successfully.
Every raw file has a matching processed CSV file.
All processed CSV headers were read successfully.


#### Explanation :

The notebook reports a successful conversion only when:

- No conversion failed.
- Every raw .csv.gz file has a matching .csv file.
- Every processed CSV header can be read.

This final check confirms that the processed eICU tables are ready for the next stage of the project.